In [1]:
import os
import torch
import random
import numpy as np
import matplotlib
%matplotlib inline

from exp.exp_long_term_forecasting import Exp_Long_Term_Forecast
"""
TimeXer:

SS: d_model': 256, 'patch_len': 6, 'dropout': 0.39424879498670423, 'lr': 0.0010101861513518219,
'batch_size': 8, 'd_ff': 512, 'e_layers': 2, 'seq_len': 6, 'pred_len': 6, 

log1p: 'd_model': 256, 'patch_len': 6, 'dropout': 0.025928388760553256, 'lr': 0.0008467287653204573,
'batch_size': 8, 'd_ff': 512, 'e_layers': 3, 'seq_len': 12, 'pred_len': 6,

Box-cox: {'d_model':192 , 'patch_len': 6, 'dropout': 0.08717833803068564, 'lr': 0.001342195567736814, 
'batch_size': 8, 'd_ff':16 , 'e_layers': 2, 'seq_len':12 , 'pred_len': 6, - 192 12 2 16

Yeo-Johnson: d_model': 256, 'patch_len': 6, 'dropout': 0.40625363288127014, 'lr': 0.00201312485183867, 
'batch_size': 8, 'd_ff': 2048, 'e_layers': 3, 'seq_len': 12, 'pred_len': 6
              
Sqrt: 'd_model': 192, 'patch_len': 6, 'dropout': 0.23893950045103568, 'lr': 0.002211997810682128, 
'batch_size': 8, 'd_ff': 16, 'e_layers': 3, 'seq_len': 7, 'pred_len': 6,
              
Log1p + 1st Diff: 'd_model': 64, 'patch_len': 6, 'dropout': 0.39828640328589804, 'lr': 0.0001860885942981913,
'batch_size': 16, 'd_ff': 16, 'e_layers': 2, 'seq_len': 20, 'pred_len': 6,

Log1p + S. Diff: 'd_model': 256, 'patch_len': 6, 'dropout': 0.22996458741459191, 'lr': 0.0006370440446688316,
'batch_size': 16, 'd_ff': 128, 'e_layers': 3, 'seq_len': 15, 'pred_len': 6,

Optimal-box-cox:
{
  "batch_size": 16,
  "d_ff": 512,
  "d_model": 256,
  "data_path": "covid.csv",
  "dropout": 0.0670541147476329,
  "e_layers": 2,
  "lr": 0.00310806844076205,
  "patch_len": 6,
  "pred_len": 6,
  "seq_len": 9
}

iTransformer:

SS: {'d_model': 256, 'patch_len': 6, 'dropout': 0.48488898001658104, 'lr': 0.0027693694524062768,
'batch_size': 8, 'd_ff': 128, 'e_layers': 1, 'seq_len': 7, 'pred_len': 6 - for optimal-box-cox too

log1p: 256 7 2 192 (8 BS)

Yeo-johnson: 192 10 3 256 (16 BS)

Sqrt: 192 15 1 256(16 BS)

log1p+1st diff: 256 20 1 192

log1p+S.diff: 256 12 1 192

box-cox: 
{
d_model = 256
patch_len = 6
dropout = 0.4769527705959334
learning_rate = 0.0007843931009740514
batch_size = 8
d_ff =16
e_layers = 3
seq_len = 9
}

"""

if __name__ == '__main__':
    for dm in [256]:
        for sl in [7]:
            for el in [1]:
                for d_ff in [128]:
                    print(dm,sl,el,d_ff)
                    class Configs(object):
                        # general
                        task_name = 'long_term_forecast'   # [long_term_forecast, short_term_forecast, imputation, classification, anomaly_detection]
                        is_training = 1
                        model_id = 'test'
                        model = "iTransformer"               # [Autoformer, Transformer, TimesNet]

                        # data loader
                        data = 'custom'
                        root_path = './dataset/'
                        data_path = 'covid.csv'
                        features = 'M'                     # [M, S, MS]
                        target = 'new_deaths'                      # only for S or MS task
                        target_index = 0
                        freq = 'h'                         # [s, t, h, d, b, w, m, …]
                        checkpoints = './checkpoints/'

                        # forecasting task
                        d_model = dm
                        patch_len = 6
                        dropout =0.48488898001658104
                        learning_rate =  0.0027693694524062768
                        batch_size = 8
                        d_ff = d_ff
                        e_layers = el
                        seq_len = sl


                        scale_method = "standardscaler"
                        difference = False
                        difforder = 'First'      #['First','Seasonal']
                        seasonal = 10
                        standardize = False 
                        standardize_diff = False 

                        subtract_last = 0

                        per_h_enable = False 
                        per_h_cutoff = 6 
                        blend_tail_steps = 3 
                        input_blend = False 
                        blend_mode = None

                        pred_len = 6
                        label_len = 4 

                        factor = 3
                        enc_in = 7
                        dec_in = 7
                        c_out = 7
                        des = 'Exp'

                        itr = 1

                        seasonal_patterns = 'Monthly'
                        inverse = True

                        # optimization
                        num_workers = 10

                        train_epochs = 10
                        patience = 3

                        loss = 'MSE'
                        lradj = 'type1'
                        use_amp = False

                        # imputation task
                        mask_rate = 0.25

                        # anomaly detection task
                        anomaly_ratio = 0.25

                        # model define
                        expand = 2                # Mamba
                        d_conv = 4                # Mamba
                        top_k = 5                  # TimesBlock
                        num_kernels = 6            # Inception
                        n_heads = 8
                        d_layers = 1
                        moving_avg = 25

                        distil = True

                        embed = 'timeF'            # [timeF, fixed, learned]
                        activation = 'gelu'
                        output_attention = False
                        channel_independence = 1   # 0: dependent, 1: independent (FreTS)
                        decomp_method = 'moving_avg'  # [moving_avg, dft_decomp]
                        use_norm = 1
                        down_sampling_layers = 0
                        down_sampling_window = 1
                        down_sampling_method = None    # [avg, max, conv]
                        seg_len = 48                   # for SegRNN

                        # GPU
                        use_gpu = True
                        gpu = 0
                        use_multi_gpu = False
                        devices = '0,1,2,3'

                        # de-stationary projector params
                        p_hidden_dims = [128, 128]
                        p_hidden_layers = 2

                        # metrics
                        use_dtw = False

                        # augmentation
                        augmentation_ratio = 0
                        seed = 2
                        jitter = False
                        scaling = False
                        permutation = False
                        randompermutation = False
                        magwarp = False
                        timewarp = False
                        windowslice = False
                        windowwarp = False
                        rotation = False
                        spawner = False
                        dtwwarp = False
                        shapedtwwarp = False
                        wdba = False
                        discdtw = False
                        discsdtw = False
                        extra_tag = ""




                    args = Configs()

                    # random seed
                    fix_seed = 2021
                    random.seed(fix_seed)
                    torch.manual_seed(fix_seed)
                    np.random.seed(fix_seed)

                    args.use_gpu = True if torch.cuda.is_available() and args.use_gpu else False

                    if args.use_gpu and args.use_multi_gpu:
                        args.dvices = args.devices.replace(' ', '')
                        device_ids = args.devices.split(',')
                        args.device_ids = [int(id_) for id_ in device_ids]
                        args.gpu = args.device_ids[0]

                    Exp = Exp_Long_Term_Forecast

                    if args.is_training:
                        for ii in range(args.itr):
                            # setting record of experiments
                            setting = '{}_{}_{}_ft{}_sl{}_ll{}_pl{}_dm{}_nh{}_el{}_dl{}_df{}_fc{}_eb{}_dt{}_{}_{}'.format(
                                args.model_id,
                                args.model,
                                args.data,
                                args.features,
                                args.seq_len,
                                args.label_len,
                                args.pred_len,
                                args.d_model,
                                args.n_heads,
                                args.e_layers,
                                args.d_layers,
                                args.d_ff,
                                args.factor,
                                args.embed,
                                args.distil,
                                args.des,ii)

                            exp = Exp(args)  # set experiments
            #                 print('>>>>>>>start training : {}>>>>>>>>>>>>>>>>>>>>>>>>>>'.format(setting))
                            exp.train(setting)

            #                 print('>>>>>>>testing : {}<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<'.format(setting))
                            exp.test(setting)


                            torch.cuda.empty_cache()
                    else:
                        ii = 0
                        setting = '{}_{}_{}_ft{}_sl{}_ll{}_pl{}_dm{}_nh{}_el{}_dl{}_df{}_fc{}_eb{}_dt{}_{}_{}'.format(args.model_id,
                                                                                                                    args.model,
                                                                                                                    args.data,
                                                                                                                    args.features,
                                                                                                                    args.seq_len,
                                                                                                                    args.label_len,
                                                                                                                    args.pred_len,
                                                                                                                    args.d_model,
                                                                                                                    args.n_heads,
                                                                                                                    args.e_layers,
                                                                                                                    args.d_layers,
                                                                                                                    args.d_ff,
                                                                                                                    args.factor,
                                                                                                                    args.embed,
                                                                                                                    args.distil,
                                                                                                                    args.des, ii)

                        exp = Exp(args)  # set experiments
                        print('>>>>>>>testing : {}<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<'.format(setting))
                        exp.test(setting, test=1)
                        torch.cuda.empty_cache()

256 7 1 128
Use GPU: cuda:0
per_h_enable: False , per_h_cutoff: 6 , input_blend: False , blend_mode: None , blend_tail_steps: 3
standardscaler
train 69
standardscaler
val 7
standardscaler
test 18
Validation loss decreased (inf --> 0.114040).  Saving model ...
Updating learning rate to 0.0027693694524062768
EarlyStopping counter: 1 out of 3
Updating learning rate to 0.0013846847262031384
Validation loss decreased (0.114040 --> 0.111947).  Saving model ...
Updating learning rate to 0.0006923423631015692
EarlyStopping counter: 1 out of 3
Updating learning rate to 0.0003461711815507846
EarlyStopping counter: 2 out of 3
Updating learning rate to 0.0001730855907753923
EarlyStopping counter: 3 out of 3
Early stopping
standardscaler
test 18
test shape: (18, 6, 7) (18, 6, 7)
test shape: (18, 6, 7) (18, 6, 7)
mse:6.577280044555664, mae:1.151940107345581

INVERSE TRANSFORMED RESULTS:

[[ 8256.      9002.     11667.      9170.     10594.     13844.    ]
 [ 9002.     11667.      9170.     10594.   